In [1]:
import pandas as pd
import openai
import os
from dotenv import load_dotenv

# .env 파일에 정의된 환경 변수를 로드합니다.
load_dotenv()

# 로드된 환경 변수에서 API 키를 가져옵니다.
upstage_api_key = os.getenv("UPSTAGE_API_KEY")

if upstage_api_key:
    print("✅ Upstage API 키를 성공적으로 로드했습니다.")
else:
    print("🚨 오류: .env 파일에서 UPSTAGE_API_KEY를 찾을 수 없습니다.")

✅ Upstage API 키를 성공적으로 로드했습니다.


In [2]:
# Upstage API와 통신할 클라이언트를 초기화합니다.
# 제공해주신 예시 코드를 기반으로 설정합니다.
client = openai.OpenAI(
    base_url="https://api.upstage.ai/v1",
    api_key=upstage_api_key,
)

print("✅ Upstage API 클라이언트가 성공적으로 초기화되었습니다.")

✅ Upstage API 클라이언트가 성공적으로 초기화되었습니다.


In [4]:
try:
    df_rfm = pd.read_csv('data/customer_rfm.csv')
    df_taste = pd.read_csv('data/customer_taste_clusters.csv')
    df_orders = pd.read_csv('data/customer_orders.csv')
    
    # CustomerID를 기준으로 RFM 세그먼트와 취향 클러스터를 병합합니다.
    df_segments = pd.merge(df_rfm, df_taste, on='CustomerID')
    
    print("데이터 파일 로드 및 병합 완료.")
    display(df_segments.head())
    
except FileNotFoundError as e:
    print(f"🚨 오류: 필수 데이터 파일이 없습니다 - {e}")

데이터 파일 로드 및 병합 완료.


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Taste_Cluster
0,C0001,86,4,180000,3,2,3,323,잠재 고객 (Potential),2
1,C0002,99,5,75000,3,3,1,331,충성 고객 (Loyal),0
2,C0003,199,4,121000,2,2,2,222,이탈 위험 고객 (At Risk),0
3,C0004,170,2,71000,2,1,1,211,이탈 위험 고객 (At Risk),3
4,C0005,116,4,155000,3,2,3,323,잠재 고객 (Potential),2


In [5]:
# 각 세그먼트의 정량적 특성을 요약합니다.
segment_summary = df_orders.merge(df_segments, on='CustomerID').groupby(['Segment', 'Taste_Cluster']).agg(
    customer_count=('CustomerID', 'nunique'),
    avg_monetary=('Price', lambda x: x.sum() / x.nunique()),
    top_categories=('ProductCategory', lambda x: x.value_counts().nlargest(2).index.tolist())
).reset_index()

print("마이크로 세그먼트 요약 테이블 생성 완료:")
display(segment_summary)

마이크로 세그먼트 요약 테이블 생성 완료:


,Segment,Taste_Cluster,customer_count,avg_monetary,top_categories
0,VIP,0,48,152833.333333,"[가공식품, 건강기능식품]"
1,VIP,1,69,267657.894737,"[건강기능식품, 가공식품]"
2,VIP,2,29,110432.835821,"[뷰티, 가공식품]"
3,VIP,3,52,189154.929577,"[생활용품, 건강기능식품]"
4,이탈 위험 고객 (At Risk),0,66,82086.206897,"[가공식품, 건강기능식품]"
5,이탈 위험 고객 (At Risk),1,67,123561.643836,"[건강기능식품, 가공식품]"
6,이탈 위험 고객 (At Risk),2,55,105155.172414,"[뷰티, 건강기능식품]"
7,이탈 위험 고객 (At Risk),3,64,95508.474576,"[생활용품, 건강기능식품]"
8,잠재 고객 (Potential),0,84,150928.571429,"[가공식품, 건강기능식품]"
9,잠재 고객 (Potential),1,99,252697.368421,"[건강기능식품, 가공식품]"


In [16]:
def generate_persona(segment_data):
    """세그먼트 데이터를 바탕으로 Upstage API를 호출하여 페르소나를 생성하는 함수"""
    prompt = f"""
너는 15년차 시니어 마케팅 분석가이자 크리에이티브 카피라이터다. 아래 데이터를 바탕으로, 해당 고객 세그먼트를 대표하는 구체적인 페르소나 1명을 생성하라.

### 고객 세그먼트 데이터
- **가치 그룹:** {segment_data['Segment']}
- **취향 그룹:** 클러스터 {segment_data['Taste_Cluster']}
- **고객 수:** {segment_data['customer_count']}명
- **1인당 평균 구매액:** 약 {int(segment_data['avg_monetary']):,}원
- **주요 구매 카테고리:** {', '.join(segment_data['top_categories'])}

### 생성할 페르소나 항목
1.  **[요약]**: 페르소나의 이름, 나이, 직업, 소득, 라이프스타일, 가치관 요약.
2.  **[목표와 동기]**: 개인적, 직업적 목표와 동기.
3.  **[페인 포인트]**: 일상과 쇼핑의 어려움, 스트레스.
4.  **[주요 정보 채널]**: 제품 정보 획득 및 여가 채널.
5.  **[마케팅 메시지 제안]**: 광고 헤드라인 1개와 짧은 설명.

결과는 위 항목에 맞춰 명확히 구분해서 작성하라.
"""
    try:
        response = client.chat.completions.create(
            model="solar-pro2", # 제공해주신 모델명으로 변경
            messages=[
                {"role": "system", "content": "You are a helpful marketing expert."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"🚨 API 호출 중 오류 발생: {e}"

print("✅ 'generate_persona' 함수가 정의되었습니다.")

✅ 'generate_persona' 함수가 정의되었습니다.


In [17]:
def run_conjoint_simulation(persona_description, product_options):
    """생성된 페르소나에게 가상 신제품 옵션을 제시하고 선택을 유도하는 함수"""
    options_text = ""
    for i, option in enumerate(product_options, 1):
        options_text += f"\n--- 선택지 {i} ---\n"
        for key, value in option.items():
            options_text += f"- {key}: {value}\n"
    
    prompt = f"""
당신은 이제부터 아래에 묘사된 페르소나에 완벽하게 빙의해서 답변해야 한다.

### 페르소나 설명
{persona_description}

### 상황
당신은 최근 부쩍 피로감을 느껴 새로운 비타민 영양제를 구매하려고 한다. 온라인 쇼핑몰에서 아래 3가지 최종 후보를 발견했다. 가격과 품질, 당신의 라이프스타일과 가치관에 가장 적합한 제품은 무엇인가?

### 신제품 선택지
{options_text}

### 질문
1.  당신이 최종적으로 선택할 제품은 몇 번 선택지인가?
2.  그 제품을 선택한 가장 결정적인 이유를 당신의 입장에서 한두 문장으로 설명하라.
"""
    try:
        response = client.chat.completions.create(
            model="solar-pro2", # 제공해주신 모델명으로 변경
            messages=[
                {"role": "system", "content": "You are acting as a specific persona."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3, # 일관된 답변을 위해 온도를 낮춤
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"🚨 API 호출 중 오류 발생: {e}"

print("✅ 'run_conjoint_simulation' 함수가 정의되었습니다.")

✅ 'run_conjoint_simulation' 함수가 정의되었습니다.


In [18]:
# 분석할 샘플 세그먼트 선택 (VIP 고객 중 가장 수가 많은 그룹)
if not segment_summary.empty:
    sample_segment = segment_summary[segment_summary['Segment'] == 'VIP'].sort_values('customer_count', ascending=False).iloc[0]
    print(f"--- 🚀 분석 대상 세그먼트 ---\n{sample_segment}\n")

    # 1. 페르소나 생성
    print("--- ⏳ 1. 페르소나 생성 요청 중... ---")
    generated_persona_text = generate_persona(sample_segment)
    print("\n--- ✅ 1. 페르소나 생성 완료 ---")
    print(generated_persona_text)

    # 2. 컨조인트 시뮬레이션
    print("\n" + "="*50 + "\n")
    print("--- ⏳ 2. 생성된 페르소나 기반 컨조인트 시뮬레이션 요청 중... ---")

    product_options = [
        {'제품명': '데일리 에너지업', '가격': '25,000원', '핵심성분': '비타민B군 100mg', '인증': '일반 식약처 인증', '형태': '알약'},
        {'제품명': '프리미엄 내추럴 비타민', '가격': '42,000원', '핵심성분': '유기농 과일 추출 비타민', '인증': '미국 USDA 유기농 인증', '형태': '액상 스틱'},
        {'제품명': '메가도스 파워 부스트', '가격': '35,000원', '핵심성분': '비타민C 2000mg + 아연', '인증': 'GMP 인증', '형태': '분말'}
    ]

    conjoint_result = run_conjoint_simulation(generated_persona_text, product_options)
    print("\n--- ✅ 2. 시뮬레이션 완료 ---")
    print(conjoint_result)
    
else:
    print("🚨 요약된 세그먼트가 없어 분석을 실행할 수 없습니다.")

--- 🚀 분석 대상 세그먼트 ---
Segment                      VIP
Taste_Cluster                  1
customer_count                69
avg_monetary       267657.894737
top_categories    [건강기능식품, 가공식품]
Name: 1, dtype: object

--- ⏳ 1. 페르소나 생성 요청 중... ---

--- ✅ 1. 페르소나 생성 완료 ---
### 1. [요약]  
- **이름**: 이민수  
- **나이**: 48세  
- **직업**: 중견 기업 임원(IT 기업 CTO)  
- **소득**: 연소득 1억 2천만 원  
- **라이프스타일**: 건강 관리에 적극적이고, 효율적인 시간 활용을 중시. 프리미엄 제품과 맞춤형 서비스를 선호하며, 주말에는 가족과 함께 하이킹이나 골프 라운딩을 즐긴다.  
- **가치관**: "건강은 최고의 투자" / "품질과 신뢰를 위해 프리미엄 비용을 지불할 가치가 있다"  

---

### 2. [목표와 동기]  
- **개인적 목표**: 50대 이후에도 활력 있는 신체 유지, 가족과의 건강한 여가 생활.  
- **직업적 목표**: 리더십 유지와 업무 효율성을 위한 에너지 관리.  
- **구매 동기**:  
  - 과학적 근거가 있는 건강기능식품으로 노화 방지.  
  - 신뢰할 수 있는 브랜드의 가공식품으로 시간 절약형 식사 해결.  

---

### 3. [페인 포인트]  
- **일상**: 높은 업무 강도로 인한 스트레스와 피로 누적.  
- **쇼핑**:  
  - "비슷한 기능성 제품 중 진짜 효과가 있는 것을 고르는 데 시간 소모"  
  - "VIP 혜택(맞춤 상담, 빠른 배송 등)이 부족해 불편함"  
- **추가 고민**: "수입 제품의 경우 국내 유통사 가격이 과도하게 부풀려진 것 같다"  

---

### 4. [주요 정보 채널]  
- **제품 정보**: 전문가 리뷰(의학

In [20]:
# 분석할 샘플 세그먼트 선택 (VIP 고객 중 가장 수가 많은 그룹)
if not segment_summary.empty:
    sample_segment = segment_summary[segment_summary['Segment'] == 'VIP'].sort_values('customer_count', ascending=False).iloc[0]
    print(f"--- 🚀 분석 대상 세그먼트 ---\n{sample_segment}\n")

    # 1. 페르소나 생성
    print("--- ⏳ 1. 페르소나 생성 요청 중... ---")
    generated_persona_text = generate_persona(sample_segment)
    print("\n--- ✅ 1. 페르소나 생성 완료 ---")
    print(generated_persona_text)

    # 2. 컨조인트 시뮬레이션
    print("\n" + "="*50 + "\n")
    print("--- ⏳ 2. 생성된 페르소나 기반 컨조인트 시뮬레이션 요청 중... ---")

    product_options = [
        {'제품명': '데일리 에너지업', '가격': '25,000원', '핵심성분': '비타민B군 100mg', '인증': '일반 식약처 인증', '형태': '알약'},
        {'제품명': '프리미엄 내추럴 비타민', '가격': '42,000원', '핵심성분': '유기농 과일 추출 비타민', '인증': '미국 USDA 유기농 인증', '형태': '액상 스틱'},
        {'제품명': '메가도스 파워 부스트', '가격': '35,000원', '핵심성분': '비타민C 2000mg + 아연', '인증': 'GMP 인증', '형태': '분말'}
    ]

    conjoint_result = run_conjoint_simulation(generated_persona_text, product_options)
    print("\n--- ✅ 2. 시뮬레이션 완료 ---")
    print(conjoint_result)
    
else:
    print("🚨 요약된 세그먼트가 없어 분석을 실행할 수 없습니다.")

--- 🚀 분석 대상 세그먼트 ---
Segment                      VIP
Taste_Cluster                  1
customer_count                69
avg_monetary       267657.894737
top_categories    [건강기능식품, 가공식품]
Name: 1, dtype: object

--- ⏳ 1. 페르소나 생성 요청 중... ---

--- ✅ 1. 페르소나 생성 완료 ---
### **[요약]**  
- **이름**: 이민수  
- **나이**: 42세  
- **직업**: IT 기업 임원 (CTO)  
- **소득**: 연 1억 5,000만 원 이상  
- **라이프스타일**: 바쁜 업무와 건강한 삶을 병행하기 위해 프리미엄 건강기능식품과 편의성 높은 가공식품을 선호. 주말에는 가족과의 시간을 중요시하며, 시간 효율성을 극대화하려는 성향.  
- **가치관**: "품질과 과학적 검증"을 신뢰하며, 지속 가능한 프리미엄 제품을 추구.  

---

### **[목표와 동기]**  
1. **개인적 목표**:  
   - 스트레스 관리와 체력 유지로 가족과의 건강한 일상 유지.  
   - 바쁜 일정 속에서도 영양 균형을 맞추기 위한 효율적인 솔루션 탐색.  
2. **직업적 동기**:  
   - 업무 성과 유지를 위해 집중력과 피로 회복에 도움되는 제품 탐색.  
   - 신뢰할 수 있는 브랜드와의 장기적 관계를 통해 시간/노력 절약.  

---

### **[페인 포인트]**  
- **일상적 어려움**:  
   - 과도한 업무 스트레스로 인한 수면 부족과 피로 누적.  
   - 외식이나 일반 가공식품에 대한 불신 ("첨가물 걱정").  
- **쇼핑 스트레스**:  
   - 제품 성분 표기나 효능의 모호함으로 인한 선택 장애.  
   - 직접 매장 방문 없이 신뢰할 수 있는 구매 채널 부재.  

---

### **[주요 정보 채널]**  
- **제